# Prototype – analysis: does removing the leading silence change the AASIST score?

This notebook compares the AASIST scores of the selected ASVspoof 2019 LA eval recordings before
and after their leading silence was removed.

**How to use it**

- Run the cells from top to bottom.
- Notebook reads files in `results/prototype/` – the trimming info saved by notebook 06 and the AASIST model scores that are the results of running `score_aasist.py`.

**Steps**

1. Setup – paths and settings
2. Results from the pod
3. Analysis – score change, decisions, EER

### Step 1 – Setup: paths and settings

**What it does:** Loads the libraries, finds the project folder and sets the path to the results
folder (`results/prototype`) and the silence threshold whose trimmed recordings were scored on
the pod.

**Why:** All the next cells read their input from this folder. `THRESHOLD_DB` must be the same
as the one chosen in notebook 06 – it selects the files `trim_info_-30dB.csv` and
`aasist_trimmed_-30dB.csv`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("pyproject.toml not found")


REPO_ROOT = find_repo_root(Path.cwd())
RESULTS_DIR = REPO_ROOT / "results" / "prototype"

THRESHOLD_DB = -30.0

TRIM_INFO_PATH = RESULTS_DIR / f"trim_info_{THRESHOLD_DB:+.0f}dB.csv"
SCORES_ORIGINAL_PATH = RESULTS_DIR / "aasist_original.csv"
SCORES_TRIMMED_PATH = RESULTS_DIR / f"aasist_trimmed_{THRESHOLD_DB:+.0f}dB.csv"
TIMINGS_PATH = RESULTS_DIR / "aasist_timings.csv"

print("Results folder:", RESULTS_DIR.relative_to(REPO_ROOT), "| exists:", RESULTS_DIR.exists())

### Step 2 – Results from the pod

**What it does:** Checks that the input files are in `results/prototype/` and shows the
timings. The scores were computed with `scripts/prototype/score_aasist.py` that produced `aasist_original.csv`,
`aasist_trimmed_-30dB.csv` (one row per recording) and `aasist_timings.csv`. The trimming info
`trim_info_-30dB.csv` comes from notebook 06.

**Why:** to check if files needed for the evaluation functions are present.

In [ ]:
for path in [TRIM_INFO_PATH, SCORES_ORIGINAL_PATH, SCORES_TRIMMED_PATH, TIMINGS_PATH]:
    print(path.relative_to(REPO_ROOT), "| exists:", path.exists())

timings = pd.read_csv(TIMINGS_PATH)
display(timings.T.rename(columns={0: "value"}))

### Step 3 – Analysis

#### 3.1 Join the tables

**What it does:** Joins, for every recording, the trimming info at `THRESHOLD_DB` (class, attack,
removed silence) with the AASIST scores before and after trimming. The score is the bona fide
logit (higher = more likely real). It adds the change `delta = score_after − score_before`, the
model's decision before and after (the class with the larger logit), whether the decision
changed, and whether the recording is shorter than the AASIST input window (~4 s), in which
case it is repeated to fill the window. The table shows the duration of every
recording before and after trimming and is saved to
`results/prototype/aasist_per_file_-{THRESHOLD_DB}dB.csv`.

**Why:** Merging collected data to store it in one table. All the next numbers and the plot come from this one table. `merge_scores` also checks
that the score files contain exactly the selected recordings and that their durations match
the trimming info – a score file from another threshold is caught here.

In [ ]:
from xai_audio_deepfake_shortcuts_audit.prototype.merge_scores import merge_scores

trim_info = pd.read_csv(TRIM_INFO_PATH)
scores_original = pd.read_csv(SCORES_ORIGINAL_PATH)
scores_trimmed = pd.read_csv(SCORES_TRIMMED_PATH)

table = merge_scores(trim_info, scores_original, scores_trimmed)

per_file_path = RESULTS_DIR / f"aasist_per_file_{THRESHOLD_DB:+.0f}dB.csv"
table.to_csv(per_file_path, index=False)
print("Saved:", per_file_path.relative_to(REPO_ROOT))

columns = [
    "file_name",
    "label",
    "attack",
    "original_s",
    "removed_s",
    "trimmed_s",
    "score_before",
    "score_after",
    "delta",
    "decision_before",
    "decision_after",
]
display(table[columns].head(10).round(2))

#### 3.2 Score change and decision changes per class

**What it does:** Shows per class the median score before and after trimming, the mean and median
change and the number of changed decisions. The second table shows the decisions in detail:
rows are the true class and the decision before trimming, columns the decision after.

**Why:** This is the main question of the prototype – to see if the model used the leading silence as a
cue, removing it should lower the scores and change the decisions to their opposites.

In [ ]:
summary = table.groupby("label").agg(
    n=("delta", "size"),
    score_before_median=("score_before", "median"),
    score_after_median=("score_after", "median"),
    delta_mean=("delta", "mean"),
    delta_median=("delta", "median"),
    removed_s_median=("removed_s", "median"),
    decision_changed=("decision_changed", "sum"),
)
display(summary.round(2))

decisions = pd.crosstab(
    [table["label"], table["decision_before"]],
    table["decision_after"],
    rownames=["true class", "decision before"],
    colnames=["decision after"],
)
display(decisions)

#### 3.3 EER before and after

**What it does:** Computes the equal error rate (EER) on the original recordings and on the
trimmed ones: the error rate at the threshold where the share of rejected bona fide equals the
share of accepted spoof. 0 % = the scores separate the two classes perfectly.

**Why:** To see whether the model can still tell real recordings from fake ones once the silence
is gone.

In [ ]:
from xai_audio_deepfake_shortcuts_audit.prototype.compute_eer import compute_eer

eer_before = compute_eer(table["score_before"], table["label"])
eer_after = compute_eer(table["score_after"], table["label"])
print(f"EER before trimming: {eer_before:.1%}")
print(f"EER after trimming:  {eer_after:.1%}")

#### 3.4 Score change vs. removed silence

**What it does:** Draws one point per recording: the removed silence on the x-axis and the
score change on the y-axis, bona fide and spoof in different colors and shapes. Recordings
whose decision changed have a dark ring. The plot is saved to
`results/prototype/aasist_delta_vs_silence_-30dB.png`.

**Why:** It shows whether the score drops more when more silence is removed, and whether the
two classes simply differ in how much silence they had.

In [ ]:
import matplotlib.pyplot as plt

from xai_audio_deepfake_shortcuts_audit.prototype.plot_delta_vs_silence import (
    plot_delta_vs_silence,
)

title = f"AASIST – score change after removing the leading silence ({THRESHOLD_DB:.0f} dB)"
out_png = RESULTS_DIR / f"aasist_delta_vs_silence_{THRESHOLD_DB:+.0f}dB.png"
fig = plot_delta_vs_silence(table, title, out_png)
display(fig)
plt.close(fig)
print("Saved:", out_png.relative_to(REPO_ROOT))

#### 3.5 Caveat – repeated recordings

**What it does:** Shows per class how the trimming changed the way AASIST sees each recording.
AASIST always gets ~4 s (64600 samples); a shorter recording is repeated until it fills this
window (the `pad()` function of the AASIST repository). Each recording falls into one column:

| Column | Before trimming | After trimming | What it means |
|---|---|---|---|
| fits → fits | ≥ 4.04 s, not repeated | ≥ 4.04 s, not repeated | only the silence is gone – a clean comparison |
| fits → repeated | ≥ 4.04 s, not repeated | < 4.04 s, repeated | the repetition appears only because of the trimming |
| repeated → repeated | < 4.04 s, repeated | < 4.04 s, repeated more often | it was repeated already (with its silence), now more copies fit |

**Why:** Removing the silence changes two things at once: there is no silence at the start, and
a short recording is repeated more often. Only for "fits → fits" can the score change be
attributed to the silence alone.


In [ ]:
from xai_audio_deepfake_shortcuts_audit.prototype.merge_scores import AASIST_WINDOW_S


def window_status(repeated: pd.Series) -> pd.Series:
    return repeated.map({False: "fits", True: "repeated"})


window = window_status(table["repeated_before"]) + " → " + window_status(table["repeated_after"])
counts = pd.crosstab(
    table["label"], window, rownames=["class"], colnames=["AASIST window: before → after trimming"]
)
display(counts)

print(f"Legend (AASIST window = {AASIST_WINDOW_S:.2f} s; shorter recordings are repeated):")
print("  fits → fits          long enough before and after – never repeated, only silence removed")
print("  fits → repeated      long enough before, too short after – repeated only after trimming")
print("  repeated → repeated  too short already before – after trimming repeated even more often")